# RQ1: transfer and stability — final results

E1 learning rates and E2 training seeds are viewed together. E1 keeps full updates with L2-SP 0.003; RQ2 retains the complete factorial. Final evaluation averages five paired outer folds within table, then weights tables equally. AUC/R² gains are CPT minus original; Brier/RMSE gains reverse direction. RMSE gains here are relative percentages.

## 0. Evidence and interpretation

The selected local evidence is a snapshot. Missing evidence remains unavailable; queued VSC jobs are not completed results.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.visualize import style, campaign as cp, questions as rq, synthesis as sy, research_views as rv
from src.visualize import choice_effects as ce, comparisons as cmp, publication as pub, dashboards as db
from src.visualize import diagnostics as dg
from src.visualize import training_summary as ts, pilots as pv, finishing as finish, prediction_diagnostics as pred
from src.visualize.figures import FigureSaver
from src.visualize.inputs import source_description
style.apply()
sink = FigureSaver('rq1/02_results')
report = cp.NotebookReport('RQ1: transfer and stability — final results', sink=sink)
print(source_description())
report.add('0. Evidence and interpretation', source_description() + '\n' + 'E1 learning rates and E2 training seeds are viewed together. E1 keeps full updates with L2-SP 0.003; RQ2 retains the complete factorial. Final evaluation averages five paired outer folds within table, then weights tables equally. AUC/R² gains are CPT minus original; Brier/RMSE gains reverse direction. RMSE gains here are relative percentages.')

## 1. Coverage, protocols and unique evidence

Training completion and benchmark completion are separate. All results retain their own matched original checkpoint. The current inference-policy table governs cross-study TabICL comparisons; a pending FP32 refresh is not published evidence.

In [ ]:
QUESTION = 1
studies = {track: rq.load(QUESTION, track) for track in ('pd', 'lgd')}
catalogs = {}
for track, values in studies.items():
    report.table(track.upper() + ' unique trial inventory', rv.inventory_summary(rv.unique_training_inventory(values)))
    report.table(track.upper() + ' recipe and inference comparability', sy.comparability_table(values))
    for e, arm, run in values:
        report.table(track.upper() + ' ' + str(run.cfg.run_name) + ' ' + str(arm or '') + ' five-fold coverage', cmp.benchmark_coverage(run))
    catalogs[track] = {(domain, metric): rq.catalog(values, QUESTION, metric, domain, reference_only=QUESTION == 2)
                       for domain in ('credit', 'ood') for metric in (sy.PRIMARY[track], sy.SECONDARY[track])}
report.add('1. Coverage, protocols and unique evidence', 'Training completion and benchmark completion are separate. All results retain their own matched original checkpoint. The current inference-policy table governs cross-study TabICL comparisons; a pending FP32 refresh is not published evidence.')

## 2. Original-to-CPT effects across the research question

Held-out credit transfer (primary and secondary metric) and non-credit retention share one view. Every dot is a table, the diamond is the equal-table mean and the bar is the table interquartile range: heterogeneity, not a confidence interval. The same tables contribute to every displayed recipe within each model and column.

In [ ]:
for track in studies:
    columns = rq.effect_columns(catalogs[track], track)
    cp.show(sink, rq.plot_effects(columns, track, title='Transfer and retention by learning rate and seed', name='effects', min_arms=1), track=track)
    for metric in (sy.PRIMARY[track], sy.SECONDARY[track]):
        report.table(track.upper() + ' original/CPT scores and paired effects: ' + metric, sy.summarize(catalogs[track]['credit', metric]))
report.add('2. Original-to-CPT effects across the research question', 'Held-out credit transfer (primary and secondary metric) and non-credit retention share one view. Every dot is a table, the diamond is the equal-table mean and the bar is the table interquartile range: heterogeneity, not a confidence interval. The same tables contribute to every displayed recipe within each model and column.')

## 3. What changes when a specific choice changes?

E2 compares training seeds 42 and 43 at the predefined reference recipe, on the same tables and folds; both seeds appear as adjacent rows of the effect figure, and their named-table differences are tabulated here. A positive contrast can mean less damage, not positive transfer.

In [ ]:
for track in studies:
    for metric in (sy.PRIMARY[track], sy.SECONDARY[track]):
        report.table(track.upper() + ' matched choice differences: ' + metric, rq.choice_differences(catalogs[track]['credit', metric], QUESTION))
report.add('3. What changes when a specific choice changes?', 'E2 compares training seeds 42 and 43 at the predefined reference recipe, on the same tables and folds; both seeds appear as adjacent rows of the effect figure, and their named-table differences are tabulated here. A positive contrast can mean less damage, not positive transfer.')

## 4. Non-credit retention alongside transfer

The four same-task non-credit tables are a bounded retention panel, plotted with the transfer effects above. Their four training-partition results are averaged within table before equal-table summaries.

In [ ]:
for track, values in studies.items():
    for metric in (sy.PRIMARY[track], sy.SECONDARY[track]):
        report.table(track.upper() + ' non-credit absolute scores and effects: ' + metric, sy.summarize(catalogs[track]['ood', metric]))
report.add('4. Non-credit retention alongside transfer', 'The four same-task non-credit tables are a bounded retention panel, plotted with the transfer effects above. Their four training-partition results are averaged within table before equal-table summaries.')

## 5. The fixed reference: named tables and prediction quality

The reference is fixed independently of held-out outcomes. Named-table plots and saved-prediction diagnostics show whether the mean hides heterogeneity. Signed residuals are not the theoretical model bias; table-bootstrap intervals are conditional on these fitted checkpoints.

In [ ]:
for track, values in studies.items():
    main = next(run for e, arm, run in values if e == 1)
    cp.show(sink, sy.plot_reference_tables(catalogs[track]['credit', sy.PRIMARY[track]], track), track=track)
    cp.show(sink, finish.plot_uncertainty(main), track=track)
    report.table(track.upper() + ' all reference metrics', ce.reference_profiles(main))
    if track == 'pd':
        cp.show(sink, dg.plot_calibration_overview(main), track=track)
    diagnostic, support = pred.analyze(main, reference=True)
    report.table(track.upper() + ' prediction coverage', support)
    report.table(track.upper() + ' paired prediction diagnostics', diagnostic)
    cp.show(sink, pred.plot_diagnostics(diagnostic, track, coverage=support), track=track)
    if track == 'lgd':
        cp.show(sink, finish.plot_lgd_distribution(main), track=track)
report.add('5. The fixed reference: named tables and prediction quality', 'The reference is fixed independently of held-out outcomes. Named-table plots and saved-prediction diagnostics show whether the mean hides heterogeneity. Signed residuals are not the theoretical model bias; table-bootstrap intervals are conditional on these fitted checkpoints.')

## 6. Cost, complete numerical results and conclusions

Transfer is model-, task- and recipe-dependent. Compare paired changes rather than raw scores across different inference settings. Missing model additions are explicit. These empirical questions were organized after results were visible; they are not preregistered directional hypotheses.

In [ ]:
for track, values in studies.items():
    primary = catalogs[track]['credit', sy.PRIMARY[track]]
    report.table(track.upper() + ' recorded cost', sy.costs(values))
    for (domain, metric), data in catalogs[track].items():
        report.table(track.upper() + ' all named-table pairs: ' + domain + ' ' + metric, data)
report.add('6. Cost, complete numerical results and conclusions', 'Transfer is model-, task- and recipe-dependent. Compare paired changes rather than raw scores across different inference settings. Missing model additions are explicit. These empirical questions were organized after results were visible; they are not preregistered directional hypotheses.')

## Complete text summary

Every displayed table, plotted value and paper caption, in section order.

In [ ]:
print(report.summary(sink))